In [1]:
import pandas as pd
import glob
import logging
import os
import gc
import numpy as np
from frameaxis.core import random_subset_null

In [2]:
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger(__name__)

# Null model: random subsets of the whole table

For each aspect x sentiment cell, draw `N` subsets of the same size as the cell, without replacement, from the whole table, and save their column means. The seed makes reruns identical. Samples are saved as float32 `.npy` (about 65 MB per cell for N = 10,000).

In [3]:
def bootstrapping(df, mode, COLUMNS):
    A = df[COLUMNS].values
    for aspect in df['aspect'].unique():
        for sentiment in ['positive', 'negative']:
            df_r_count = df.query('aspect == @aspect and sentiment == @sentiment').shape[0]
            logger.info("{}-{}-{}".format(mode, aspect, sentiment))
            null = random_subset_null(A, df_r_count, N, SEED)
            np.save("{}/bootstrap_{}_{}.npy".format(mode, aspect.replace("/", "-"), sentiment), null.astype(np.float32))

In [4]:
N = 10000
SEED = 2021

In [5]:
mode = "average"
logger.info(mode)
os.makedirs(mode, exist_ok=True)

df = pd.read_csv("big_table_by_{}.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

2026-11-05 10:40:41 __main__     INFO     average


1621

In [6]:
bootstrapping(df, mode, COLUMNS)

2026-11-05 10:40:42 __main__     INFO     average-service-positive


2026-11-05 10:40:43 __main__     INFO     average-service-negative


2026-11-05 10:40:44 __main__     INFO     average-food-positive


2026-11-05 10:40:45 __main__     INFO     average-food-negative


2026-11-05 10:40:47 __main__     INFO     average-anecdotes/miscellaneous-positive


2026-11-05 10:40:48 __main__     INFO     average-anecdotes/miscellaneous-negative


2026-11-05 10:40:49 __main__     INFO     average-price-positive


2026-11-05 10:40:50 __main__     INFO     average-price-negative


2026-11-05 10:40:52 __main__     INFO     average-ambience-positive


2026-11-05 10:40:53 __main__     INFO     average-ambience-negative


In [7]:
mode = "second_moment"
logger.info(mode)
os.makedirs(mode, exist_ok=True)

df = pd.read_csv("big_table_by_{}_with_corpus_mean.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

2026-11-05 10:40:54 __main__     INFO     second_moment


1621

In [8]:
bootstrapping(df, mode, COLUMNS)

2026-11-05 10:40:55 __main__     INFO     second_moment-service-positive


2026-11-05 10:40:56 __main__     INFO     second_moment-service-negative


2026-11-05 10:40:57 __main__     INFO     second_moment-food-positive


2026-11-05 10:40:59 __main__     INFO     second_moment-food-negative


2026-11-05 10:41:00 __main__     INFO     second_moment-anecdotes/miscellaneous-positive


2026-11-05 10:41:01 __main__     INFO     second_moment-anecdotes/miscellaneous-negative


2026-11-05 10:41:02 __main__     INFO     second_moment-price-positive


2026-11-05 10:41:04 __main__     INFO     second_moment-price-negative


2026-11-05 10:41:05 __main__     INFO     second_moment-ambience-positive


2026-11-05 10:41:06 __main__     INFO     second_moment-ambience-negative
